In [0]:
from src.config import (
    TEST_TABLE,
    PREDICTION_TABLE,
    REGISTERED_MODEL,
    RANDOM_STATE,
)

print("Test table       :", TEST_TABLE)
print("Prediction table :", PREDICTION_TABLE)
print("Registered model :", REGISTERED_MODEL)

In [0]:
import mlflow
import mlflow.sklearn
import pandas as pd

from pyspark.sql import functions as F

from src.feature_engineering import (
    create_features,
    validate_features,
    get_feature_columns,
)

from src.inference import (
    validate_inference_data,
    generate_predictions,
    validate_predictions,
)

print("Libraries imported successfully.")

In [0]:
client = mlflow.MlflowClient()

print("MLflow client created.")

In [0]:
champion = client.get_model_version_by_alias(
    REGISTERED_MODEL,
    "Champion",
)

champion_version = champion.version

print("==========================================")
print("CHAMPION MODEL")
print("==========================================")
print("Model name :", champion.name)
print("Version    :", champion_version)
print("Alias      : Champion")
print("==========================================")

In [0]:
model_uri = (
    f"models:/{REGISTERED_MODEL}@Champion"
)

print("Loading model:")
print(model_uri)

champion_model = mlflow.sklearn.load_model(
    model_uri
)

print("Champion model loaded successfully.")

In [0]:
inference_df = spark.table(TEST_TABLE)

print("Inference source:", TEST_TABLE)
print("Inference rows  :", inference_df.count())

display(inference_df)

In [0]:
required_input_columns = [
    "record_id",
    "sepal_length",
    "sepal_width",
    "petal_length",
    "petal_width",
    "target",
    "species",
]

missing_columns = [
    column
    for column in required_input_columns
    if column not in inference_df.columns
]

print("Required columns:")
for column in required_input_columns:
    print(" -", column)

print("\nMissing columns:", missing_columns)

assert not missing_columns, (
    f"Missing required columns: {missing_columns}"
)

print("Input schema validation: PASSED")

In [0]:
null_counts = (
    inference_df
    .select([
        F.sum(
            F.col(column).isNull().cast("int")
        ).alias(column)
        for column in required_input_columns
    ])
)

display(null_counts)

null_row = null_counts.collect()[0]

total_nulls = sum(
    value or 0
    for value in null_row
)

assert total_nulls == 0, (
    f"Inference input contains {total_nulls} null values"
)

print("Inference input quality validation: PASSED")

In [0]:
inference_feature_df = create_features(
    inference_df
)

print("Inference features created.")

display(inference_feature_df)

In [0]:
validate_features(
    inference_feature_df
)

print("Inference feature validation completed.")

In [0]:
feature_columns = get_feature_columns()

inference_pdf = (
    inference_feature_df
    .select(
        "record_id",
        *feature_columns,
        "target",
        "species",
    )
    .toPandas()
)

print("Inference DataFrame shape:")
print(inference_pdf.shape)

display(inference_pdf.head())

In [0]:
X_inference = inference_pdf[
    feature_columns
]

print("Model input shape:", X_inference.shape)

print("\nModel features:")
for column in feature_columns:
    print(" -", column)

In [0]:
validate_inference_data(
    X_inference,
    feature_columns,
)

print("Model input validation completed.")

In [0]:
prediction_pdf = generate_predictions(
    champion_model,
    inference_pdf,
    feature_columns,
)

print("Predictions generated successfully.")

display(
    prediction_pdf[
        [
            "record_id",
            "target",
            "prediction",
            "probability_setosa",
            "probability_versicolor",
            "probability_virginica",
        ]
    ]
)

In [0]:
validate_predictions(
    prediction_pdf
)

print("Prediction output validation completed.")

In [0]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

inference_accuracy = accuracy_score(
    prediction_pdf["target"],
    prediction_pdf["prediction"],
)

inference_precision = precision_score(
    prediction_pdf["target"],
    prediction_pdf["prediction"],
    average="weighted",
    zero_division=0,
)

inference_recall = recall_score(
    prediction_pdf["target"],
    prediction_pdf["prediction"],
    average="weighted",
    zero_division=0,
)

inference_f1 = f1_score(
    prediction_pdf["target"],
    prediction_pdf["prediction"],
    average="weighted",
    zero_division=0,
)

print("==========================================")
print("INFERENCE METRICS")
print("==========================================")
print(f"Accuracy : {inference_accuracy:.4f}")
print(f"Precision: {inference_precision:.4f}")
print(f"Recall   : {inference_recall:.4f}")
print(f"F1 Score : {inference_f1:.4f}")
print("==========================================")

In [0]:
from datetime import datetime, timezone

prediction_pdf["model_name"] = REGISTERED_MODEL
prediction_pdf["model_version"] = str(champion_version)
prediction_pdf["model_alias"] = "Champion"

prediction_pdf["inference_source"] = TEST_TABLE

prediction_pdf["prediction_timestamp"] = (
    datetime.now(timezone.utc)
)

print("Prediction metadata added.")

display(prediction_pdf.head())

In [0]:
prediction_output_columns = [
    "record_id",

    "sepal_length",
    "sepal_width",
    "petal_length",
    "petal_width",

    "petal_to_sepal_length_ratio",
    "petal_to_sepal_width_ratio",

    "target",
    "species",

    "prediction",

    "probability_setosa",
    "probability_versicolor",
    "probability_virginica",

    "model_name",
    "model_version",
    "model_alias",

    "inference_source",
    "prediction_timestamp",
]

prediction_pdf = prediction_pdf[
    prediction_output_columns
]

print(
    "Final prediction columns:",
    len(prediction_output_columns)
)

display(prediction_pdf)

In [0]:
prediction_spark_df = spark.createDataFrame(
    prediction_pdf
)

print("Prediction Spark DataFrame created.")

prediction_spark_df.printSchema()

In [0]:
input_count = inference_df.count()

prediction_count = prediction_spark_df.count()

print("Input rows      :", input_count)
print("Prediction rows :", prediction_count)

assert input_count == prediction_count, (
    f"Input/prediction count mismatch: "
    f"{input_count} != {prediction_count}"
)

print("Prediction count validation: PASSED")

In [0]:
duplicate_prediction_ids = (
    prediction_spark_df
    .groupBy("record_id")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_count = duplicate_prediction_ids.count()

print(
    "Duplicate prediction IDs:",
    duplicate_count
)

assert duplicate_count == 0, (
    f"Found {duplicate_count} duplicate prediction IDs"
)

print("Prediction key validation: PASSED")

In [0]:
display(
    prediction_spark_df.orderBy("record_id")
)

In [0]:
(
    prediction_spark_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(PREDICTION_TABLE)
)

print(
    "Prediction table written successfully:"
)
print(PREDICTION_TABLE)

In [0]:
saved_predictions = spark.table(
    PREDICTION_TABLE
)

print(
    "Saved prediction rows:",
    saved_predictions.count()
)

display(saved_predictions)

In [0]:
saved_count = saved_predictions.count()

assert saved_count == prediction_count, (
    f"Persisted prediction count mismatch: "
    f"{saved_count} != {prediction_count}"
)

print("Persisted prediction count validation: PASSED")

In [0]:
metadata_check = (
    saved_predictions
    .select(
        "model_name",
        "model_version",
        "model_alias",
    )
    .distinct()
)

display(metadata_check)

In [0]:
prediction_distribution = (
    saved_predictions
    .groupBy("prediction")
    .count()
    .orderBy("prediction")
)

display(prediction_distribution)

In [0]:
with mlflow.start_run(
    run_name="iris_batch_inference"
) as inference_run:

    inference_run_id = inference_run.info.run_id

    mlflow.log_param(
        "model_name",
        REGISTERED_MODEL,
    )

    mlflow.log_param(
        "model_version",
        str(champion_version),
    )

    mlflow.log_param(
        "model_alias",
        "Champion",
    )

    mlflow.log_param(
        "input_table",
        TEST_TABLE,
    )

    mlflow.log_param(
        "output_table",
        PREDICTION_TABLE,
    )

    mlflow.log_metric(
        "prediction_count",
        prediction_count,
    )

    mlflow.log_metric(
        "inference_accuracy",
        inference_accuracy,
    )

    mlflow.log_metric(
        "inference_f1",
        inference_f1,
    )

    mlflow.set_tags({
        "project": "iris-mlops",
        "pipeline_stage": "batch_inference",
        "model_alias": "Champion",
        "environment": "development",
    })

print("MLflow inference run completed.")
print("Run ID:", inference_run_id)

In [0]:
assert prediction_count > 0
assert saved_count == prediction_count
assert champion_version is not None
assert inference_run_id is not None

print("==========================================")
print("05_BATCH_INFERENCE COMPLETED SUCCESSFULLY")
print("==========================================")
print(f"Champion Model     : {REGISTERED_MODEL}")
print(f"Champion Version   : {champion_version}")
print(f"Input Table        : {TEST_TABLE}")
print(f"Output Table       : {PREDICTION_TABLE}")
print(f"Prediction Count   : {prediction_count}")
print(f"Inference Accuracy : {inference_accuracy:.4f}")
print(f"Inference F1       : {inference_f1:.4f}")
print(f"MLflow Run ID      : {inference_run_id}")
print("==========================================")